# PySpark practice 13: performance and optimization

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-practice-13-performance.ipynb)

11 exercises on skew, salting, broadcasts, pushdown and pruning, small files, exchange reuse, AQE and anti-patterns. Each exercise names the [reference entries](pyspark-reference-13-performance.ipynb) it practises, shows its inputs and the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and the helpers from [PySpark reference 13](pyspark-reference-13-performance.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:110])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

import glob
import os
import time as _time


def count_ops(df, name):
    """How many times an operator name appears in df's physical plan."""
    return sum(name in line for line in plan_lines(df))


def shuffles(df):
    """Shuffle exchanges in df's physical plan (BroadcastExchange is not a shuffle)."""
    return sum("Exchange " in line and "BroadcastExchange" not in line for line in plan_lines(df))


def timed_noop(df):
    """Run df fully without writing output (the noop sink) and return seconds taken."""
    start = _time.time()
    df.write.format("noop").mode("overwrite").save()
    return round(_time.time() - start, 2)

## Exercises

4 easy, 6 medium, 1 hard. Write your answer in the empty cell under each exercise as a DataFrame called `result`, run it, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Find the skewed keys](#exercise-1-find-the-skewed-keys) | Easy | [Detecting skew](pyspark-reference-13-performance.ipynb#detecting-skew) |
| 2 | [Two-stage salted aggregation](#exercise-2-two-stage-salted-aggregation) | Medium | [Fixing skew](pyspark-reference-13-performance.ipynb#fixing-skew) |
| 3 | [Remove a shuffle with a broadcast](#exercise-3-remove-a-shuffle-with-a-broadcast) | Easy | [Avoiding shuffles](pyspark-reference-13-performance.ipynb#avoiding-shuffles) |
| 4 | [Fix a filter that is not pushed down](#exercise-4-fix-a-filter-that-is-not-pushed-down) | Medium | [Column pruning, predicate pushdown and partition pruning](pyspark-reference-13-performance.ipynb#column-pruning-predicate-pushdown-and-partition-pruning) |
| 5 | [Confirm dynamic partition pruning](#exercise-5-confirm-dynamic-partition-pruning) | Medium | [Dynamic partition pruning](pyspark-reference-13-performance.ipynb#dynamic-partition-pruning) |
| 6 | [Collapse a withColumn loop](#exercise-6-collapse-a-withcolumn-loop) | Easy | [Patterns that silently cost time](pyspark-reference-13-performance.ipynb#patterns-that-silently-cost-time) |
| 7 | [Compact small files](#exercise-7-compact-small-files) | Medium | [Small files](pyspark-reference-13-performance.ipynb#small-files) |
| 8 | [Spot exchange reuse](#exercise-8-spot-exchange-reuse) | Medium | [Exchange reuse](pyspark-reference-13-performance.ipynb#exchange-reuse) |
| 9 | [Verify column pruning](#exercise-9-verify-column-pruning) | Easy | [Column pruning, predicate pushdown and partition pruning](pyspark-reference-13-performance.ipynb#column-pruning-predicate-pushdown-and-partition-pruning) |
| 10 | [Partitions AQE produces for a target size](#exercise-10-partitions-aqe-produces-for-a-target-size) | Hard | [Sizing shuffle partitions and AQE coalescing](pyspark-reference-13-performance.ipynb#sizing-shuffle-partitions-and-aqe-coalescing) |
| 11 | [Filter arrays before exploding](#exercise-11-filter-arrays-before-exploding) | Medium | [Patterns that silently cost time](pyspark-reference-13-performance.ipynb#patterns-that-silently-cost-time) |

## Exercise 1: Find the skewed keys

**Difficulty:** Easy · about 5 min · **Practises:** [Detecting skew](pyspark-reference-13-performance.ipynb#detecting-skew)

*Before fixing skew, prove it: measure the key distribution.*

For `events_big`, return the top 3 keys by row count as `key`, `rows` and `share` (of all rows, rounded to 3), ordered by `rows` descending.

In [ ]:
events_big = spark.range(50_000).select(F.when(F.col("id") < 30_000, 7).when(F.col("id") < 40_000, 3).otherwise(F.col("id") % 500).alias("key"))

**Expected output**

| key | rows | share |
| --- | --- | --- |
| 7 | 30020 | 0.6 |
| 3 | 10020 | 0.2 |
| 2 | 20 | 0.0 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
total = events_big.count()
result = events_big.groupBy("key").agg(F.count("*").alias("rows")).withColumn("share", F.round(F.col("rows") / total, 3)).orderBy(F.desc("rows")).limit(3)
result.show()

## Exercise 2: Two-stage salted aggregation

**Difficulty:** Medium · about 15 min · **Practises:** [Fixing skew](pyspark-reference-13-performance.ipynb#fixing-skew)

*Salting a groupBy spreads a hot key across tasks; the result must not change.*

Aggregate `sum(v)` per `key` of `skewed` in two stages with 8 salt values (salt = `id % 8`). Return the three largest keys by total as `key`, `total`, ordered by `total` descending, and make sure it equals the plain aggregation.

In [ ]:
skewed = spark.range(40_000).select("id", F.when(F.col("id") < 30_000, 0).otherwise(F.col("id") % 100).alias("key"), F.lit(1).alias("v"))

**Expected output**

| key | total |
| --- | --- |
| 0 | 30100 |
| 1 | 100 |
| 2 | 100 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
salted = (skewed.withColumn("salt", F.col("id") % 8)
          .groupBy("key", "salt").agg(F.sum("v").alias("partial"))
          .groupBy("key").agg(F.sum("partial").alias("total")))
result = salted.orderBy(F.desc("total"), "key").limit(3)
result.show()

## Exercise 3: Remove a shuffle with a broadcast

**Difficulty:** Easy · about 5 min · **Practises:** [Avoiding shuffles](pyspark-reference-13-performance.ipynb#avoiding-shuffles)

*Broadcasting the small side is the most common single performance fix.*

With automatic broadcasting disabled, count shuffle exchanges (`shuffles(df)`, which ignores `BroadcastExchange`) for `big.join(small, 'k')` with and without `F.broadcast(small)`. Return `variant` and `exchanges`. Re-enable automatic broadcasting afterwards.

In [ ]:
big = spark.range(100_000).select((F.col("id") % 100).alias("k"), "id")
small = spark.range(100).select(F.col("id").alias("k"), (F.col("id") % 3).alias("grp"))

**Expected output**

| variant | exchanges |
| --- | --- |
| sort_merge | 2 |
| broadcast | 0 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
rows = [("sort_merge", shuffles(big.join(small, "k"))), ("broadcast", shuffles(big.join(F.broadcast(small), "k")))]
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
result = spark.createDataFrame(rows, "variant string, exchanges int")
result.show()

## Exercise 4: Fix a filter that is not pushed down

**Difficulty:** Medium · about 10 min · **Practises:** [Column pruning, predicate pushdown and partition pruning](pyspark-reference-13-performance.ipynb#column-pruning-predicate-pushdown-and-partition-pruning)

*A cast on the column stops Parquet from skipping data.*

Read the Parquet at `pq` and compare two filters selecting order 9: `cast(order_id as string) = '9'` and `order_id = 9`. Return `filter` and `pushed` (whether an `EqualTo(order_id, ...)` filter appears in `PushedFilters`; the automatic `IsNotNull` filter does not count).

In [ ]:
pq = f"{WORK_DIR}/p13_orders"
orders.write.mode("overwrite").parquet(pq)

**Expected output**

| filter | pushed |
| --- | --- |
| cast_to_string | false |
| typed_comparison | true |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
def pushed(df):
    line = next(l for l in plan_lines(df, "formatted") if l.strip().startswith("PushedFilters"))
    return "EqualTo(order_id" in line

scan = spark.read.parquet(pq)
result = spark.createDataFrame([
    ("cast_to_string", pushed(scan.where(F.col("order_id").cast("string") == "9"))),
    ("typed_comparison", pushed(scan.where(F.col("order_id") == 9))),
], "filter string, pushed boolean")
result.show()

## Exercise 5: Confirm dynamic partition pruning

**Difficulty:** Medium · about 10 min · **Practises:** [Dynamic partition pruning](pyspark-reference-13-performance.ipynb#dynamic-partition-pruning)

*DPP lets a filtered dimension skip partitions of a large fact table.*

Join the fact table partitioned by `customer_id` (at `fact_path`) to customers in the `West` region. Return one row with `has_dpp` (whether the plan shows a dynamic pruning expression) and `rows` (joined rows).

In [ ]:
fact_path = f"{WORK_DIR}/p13_fact"
orders.write.mode("overwrite").partitionBy("customer_id").parquet(fact_path)

**Expected output**

| has_dpp | rows |
| --- | --- |
| true | 3 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
joined = spark.read.parquet(fact_path).join(customers.where("region = 'West'"), "customer_id")
has_dpp = any("dynamicpruning" in line for line in plan_lines(joined, "formatted"))
result = spark.createDataFrame([(has_dpp, joined.count())], "has_dpp boolean, rows int")
result.show()

## Exercise 6: Collapse a withColumn loop

**Difficulty:** Easy · about 5 min · **Practises:** [Patterns that silently cost time](pyspark-reference-13-performance.ipynb#patterns-that-silently-cost-time)

*Long withColumn loops make planning slow on wide tables.*

Build 30 derived columns `c0`..`c29` (`id * i`) on `base` with a loop of `withColumn` and with one `select`. Return `approach` and `analyzed_projects` (count of `Project [` in the analyzed logical plan).

In [ ]:
base = spark.range(5)

**Expected output**

| approach | analyzed_projects |
| --- | --- |
| withColumn_loop | 30 |
| single_select | 1 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
def analyzed_projects(df):
    text = "\n".join(plan_lines(df, "extended"))
    return text.split("== Analyzed Logical Plan ==")[1].split("== Optimized Logical Plan ==")[0].count("Project [")

looped = base
for i in range(30):
    looped = looped.withColumn(f"c{i}", F.col("id") * i)
single = base.select("id", *[(F.col("id") * i).alias(f"c{i}") for i in range(30)])
result = spark.createDataFrame([("withColumn_loop", analyzed_projects(looped)), ("single_select", analyzed_projects(single))], "approach string, analyzed_projects int")
result.show()

## Exercise 7: Compact small files

**Difficulty:** Medium · about 10 min · **Practises:** [Small files](pyspark-reference-13-performance.ipynb#small-files)

*Small files slow every downstream read; compaction rewrites them into fewer files.*

Write `order_items` with `repartition(10)` to `messy`, then compact it into exactly 2 files at `clean`. Return `stage` (`before`, `after`) and `files`.

In [ ]:
messy, clean = f"{WORK_DIR}/p13_messy", f"{WORK_DIR}/p13_clean"

**Expected output**

| stage | files |
| --- | --- |
| before | 10 |
| after | 2 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
order_items.repartition(10).write.mode("overwrite").parquet(messy)
spark.read.parquet(messy).repartition(2).write.mode("overwrite").parquet(clean)
count_files = lambda p: len(glob.glob(f"{p}/*.parquet"))
result = spark.createDataFrame([("before", count_files(messy)), ("after", count_files(clean))], "stage string, files int")
result.show()

## Exercise 8: Spot exchange reuse

**Difficulty:** Medium · about 10 min · **Practises:** [Exchange reuse](pyspark-reference-13-performance.ipynb#exchange-reuse)

*Spark computes a repeated subplan once within a query.*

Union the top and bottom customers by revenue from the same aggregated DataFrame. Return one row with `reused` (whether `ReusedExchange` appears in the final plan after running it) and `rows` (rows in the union: revenue > 400 or < 100).

In [ ]:
per_customer = orders.groupBy("customer_id").agg(F.sum("amount").alias("revenue"))

**Expected output**

| reused | rows |
| --- | --- |
| true | 3 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
both = per_customer.where("revenue > 400").unionByName(per_customer.where("revenue < 100"))
result = spark.createDataFrame([("ReusedExchange" in final_plan(both), both.count())], "reused boolean, rows int")
result.show()

## Exercise 9: Verify column pruning

**Difficulty:** Easy · about 5 min · **Practises:** [Column pruning, predicate pushdown and partition pruning](pyspark-reference-13-performance.ipynb#column-pruning-predicate-pushdown-and-partition-pruning)

*Selecting columns early lets columnar formats read less.*

Read the Parquet at `pq2` and select `order_id`, `amount` after a filter on status. Return one row with `read_columns` (the field names in `ReadSchema`, comma-separated, in order). Which column is read even though it is not selected, and why?

In [ ]:
pq2 = f"{WORK_DIR}/p13_orders2"
orders.write.mode("overwrite").parquet(pq2)

**Expected output**

| read_columns |
| --- |
| order_id,status,amount |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
q = spark.read.parquet(pq2).where("status = 'paid'").select("order_id", "amount")
schema_line = next(l for l in plan_lines(q, "formatted") if l.strip().startswith("ReadSchema"))
fields = re.findall(r"(\w+):", schema_line.split("struct<", 1)[1])
result = spark.createDataFrame([(",".join(fields),)], "read_columns string")
result.show()

## Exercise 10: Partitions AQE produces for a target size

**Difficulty:** Hard · about 15 min · **Practises:** [Sizing shuffle partitions and AQE coalescing](pyspark-reference-13-performance.ipynb#sizing-shuffle-partitions-and-aqe-coalescing)

*Understanding AQE coalescing explains task counts in the Spark UI.*

With `spark.sql.shuffle.partitions` = 40, run `groupBy('k').count()` on `data` three ways: default AQE, AQE coalescing disabled, and AQE with `spark.sql.adaptive.coalescePartitions.minPartitionSize` = `1KB` and an advisory size of `8KB`. Return `setting` and `more_than_one` (whether the result has more than one partition), in that order. Reset every setting afterwards.

In [ ]:
data = spark.range(200_000).select((F.col("id") % 20_000).alias("k"))

**Expected output**

| setting | more_than_one |
| --- | --- |
| default | false |
| coalescing_off | true |
| small_targets | true |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "40")
settings = [
    ("default", {}),
    ("coalescing_off", {"spark.sql.adaptive.coalescePartitions.enabled": "false"}),
    ("small_targets", {"spark.sql.adaptive.coalescePartitions.minPartitionSize": "1KB", "spark.sql.adaptive.advisoryPartitionSizeInBytes": "8KB"}),
]
rows = []
for label, conf in settings:
    for k, v in conf.items():
        spark.conf.set(k, v)
    g = data.groupBy("k").count()
    g.collect()
    rows.append((label, g.rdd.getNumPartitions() > 1))
    for k in conf:
        spark.conf.unset(k)
spark.conf.set("spark.sql.shuffle.partitions", "4")
result = spark.createDataFrame(rows, "setting string, more_than_one boolean")
result.show()

## Exercise 11: Filter arrays before exploding

**Difficulty:** Medium · about 10 min · **Practises:** [Patterns that silently cost time](pyspark-reference-13-performance.ipynb#patterns-that-silently-cost-time)

*Exploding first multiplies rows only to throw most of them away.*

From `carts`, return `cart_id` and `item` for items with quantity >= 3, filtering the zipped array with `filter` **before** exploding. Order by `cart_id`, `item`.

In [ ]:
carts = spark.createDataFrame([(1, ["a", "b", "c"], [1, 5, 3]), (2, ["d"], [2])], "cart_id int, items array<string>, qty array<int>")
carts.show()

**Expected output**

| cart_id | item |
| --- | --- |
| 1 | b |
| 1 | c |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
kept = F.filter(F.arrays_zip("items", "qty"), lambda x: x["qty"] >= 3)
result = carts.select("cart_id", F.explode(kept).alias("line")).select("cart_id", F.col("line.items").alias("item")).orderBy("cart_id", "item")
result.show()

**Next:** look anything up in [PySpark reference 13: performance and optimization](pyspark-reference-13-performance.ipynb), or test yourself across topics with the [PySpark coding challenges](pyspark-coding-challenges.ipynb).